# Aaron Randrianarisona - Bloc2 Programmation python pour la sécurité Projet final - Investigation d’un incident de sécurité

## Le scénario — « Opération Brume »

**Nordis Logistique** (entreprise fictive de transport : ~300 postes, 3 sites à Rouen, Lyon et Lille) a subi une attaque. Un lundi matin, plusieurs serveurs de fichiers sont devenus inaccessibles. La direction veut comprendre ce qui s’est passé, ce qui a été volé et comment éviter que cela se reproduise.

On vous remet les journaux des trois dernières semaines issus de sept sources différentes. Quelque part dans ces dizaines de milliers de lignes — noyées dans l’activité normale de l’entreprise — se trouvent les traces de l’attaque : l’entrée des attaquants, leur progression dans le réseau, le vol de données et le sabotage final.

Votre mission : reconstituer la chronologie de l’attaque, identifier les indicateurs de compromission et formuler des recommandations — uniquement à partir des données, en écrivant vos propres scripts.

### Pre sétup

In [1]:
!python -m venv .venv 
!.venv\Scripts\activate #windows
#!source .venv/bin/activate        # Linux / macOS
%pip install -r requirements.txt

Error: [Errno 13] Permission denied: 'c:\\Users\\aaron\\Documents\\Alternance\\PMN\\Projet_cyber_python_Dafnis\\Semaine 2 - Python et cybersecurite\\projet\\.venv\\Scripts\\python.exe'


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
#!/usr/bin/env python3
"""
Script de démarrage — projet « Opération Brume ».

Charge les différentes sources avec pandas et affiche quelques repères.
À vous de l'étendre pour reconstituer l'attaque. Ce fichier ne contient
AUCUNE réponse : c'est un point de départ.

    pip install pandas
    python3 demarrage.py --donnees ./donnees
"""
import argparse
import json
from pathlib import Path

import pandas as pd


def charger(dossier: Path) -> dict[str, pd.DataFrame]:
    data: dict[str, pd.DataFrame] = {}
    data["vpn"] = pd.read_csv(dossier / "vpn_auth.csv", parse_dates=["horodatage"])
    data["proxy"] = pd.read_csv(dossier / "proxy_web.csv", parse_dates=["horodatage"])
    data["firewall"] = pd.read_csv(dossier / "firewall.csv", parse_dates=["horodatage"])
    data["mail"] = pd.read_csv(dossier / "mail_gateway.csv", parse_dates=["horodatage"])
    data["inventaire"] = pd.read_csv(dossier / "inventaire_parc.csv")
    # JSON Lines : un objet par ligne
    with open(dossier / "ad_events.jsonl", encoding="utf-8") as f:
        data["ad"] = pd.DataFrame(json.loads(l) for l in f)
    data["ad"]["horodatage"] = pd.to_datetime(data["ad"]["horodatage"])
    # JSON classique (liste d'objets)
    with open(dossier / "edr_alerts.json", encoding="utf-8") as f:
        data["edr"] = pd.DataFrame(json.load(f))
    return data


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--donnees", type=Path, default=Path("donnees"))
    args = p.parse_args()
    data = charger(args.donnees)

    print("=== Volumes ===")
    for nom, df in data.items():
        print(f"  {nom:12} {len(df):>7} lignes")

    print("\n=== Période couverte (VPN) ===")
    print(" ", data["vpn"].horodatage.min(), "->", data["vpn"].horodatage.max())

    print("\n=== Exemple : connexions VPN par résultat ===")
    print(data["vpn"].resultat.value_counts().to_string())

    print("\n=== À vous de jouer ===")
    print("  Pistes : connexions depuis l'étranger ? échecs répétés puis succès ?")
    print("           comptes créés la nuit ? gros volumes sortants ? IP inconnues ?")


#if __name__ == "__main__":
#    main()


#### Charger les données

In [ ]:
path = Path("./donnees") # à remplacer par le chemin vers les données
data = charger(path)

In [5]:
data

{'vpn':                     horodatage      login        ip_source pays  \
 0    2026-08-31 07:41:07+02:00     y.roux   92.154.192.236   FR   
 1    2026-08-31 08:02:55+02:00   m.durand      88.120.7.43   FR   
 2    2026-08-31 08:15:00+02:00  d.garnier  176.150.205.132   FR   
 3    2026-08-31 08:23:49+02:00  t.garnier  176.150.143.126   FR   
 4    2026-08-31 08:24:04+02:00    l.leroy   88.120.208.104   FR   
 ...                        ...        ...              ...  ...   
 2082 2026-09-20 14:17:37+02:00   l.martin   92.154.161.167   FR   
 2083 2026-09-20 14:43:02+02:00    b.andre  176.150.127.241   FR   
 2084 2026-09-20 15:18:15+02:00   m.durand  176.150.241.159   FR   
 2085 2026-09-20 15:58:23+02:00    b.andre  176.150.152.135   FR   
 2086 2026-09-20 16:46:20+02:00   l.martin  176.150.113.137   FR   
 
      site_rattachement resultat motif  duree_min  
 0                siege   succes   NaN        106  
 1                siege   succes   NaN        244  
 2                s

In [9]:
data["proxy"].head(10)

,horodatage,login,ip_source,methode,host,url,statut,octets_envoyes,octets_recus,categorie
0,2026-08-31 07:03:12+02:00,s.garnier,PC-LIL-1161,GET,update.systeme.example,/,404,1483,12384,normal
1,2026-08-31 07:03:40+02:00,t.legrand,PC-SIE-1033,POST,www.transporteur-partenaire.example,/,200,507,9164,normal
2,2026-08-31 07:03:49+02:00,c.garnier,PC-LIL-1208,GET,intranet.nordis.example,/dossier/12,200,463,74100,normal
3,2026-08-31 07:03:57+02:00,h.bernard,PC-LYO-1227,GET,www.banque-pro.example,/recherche?q=colis,404,248,58911,normal
4,2026-08-31 07:04:19+02:00,b.mercier,PC-SIE-1089,POST,portail-rh.nordis.example,/dossier/12,304,429,58315,normal
5,2026-08-31 07:05:03+02:00,h.david,PC-SIE-1002,GET,cdn.scripts.example,/static/app.js,200,1511,54215,normal
6,2026-08-31 07:05:56+02:00,n.blanc,PC-LYO-1215,GET,erp.nordis.example,/api/data,200,447,22961,normal
7,2026-08-31 07:06:13+02:00,g.martin,PC-SIE-1087,POST,www.actualites.example,/recherche?q=colis,200,259,30844,normal
8,2026-08-31 07:07:47+02:00,n.francois,PC-SIE-1196,POST,video.formation.example,/api/data,304,294,27436,normal
9,2026-08-31 07:08:50+02:00,h.david,PC-SIE-1002,GET,www.banque-pro.example,/dossier/12,200,575,12352,normal


### Niveau 1 — Comprendre les données 

Combien d’utilisateurs, de postes et de serveurs compte l’entreprise ? Sur quelle période portent les journaux ?
À quoi ressemble une journée « normale » (volumes, horaires) ? Cette référence vous servira à repérer l’anormal.


In [ ]:
# nombre d'utilisateurs uniques
# nombre de postes uniques
# nombre de serveurs 
# date de début des logs & date de fin des logs
# etablissement d'une journée type

### Niveau 2 — Trouver le point d’entrée

Quel compte utilisateur a été compromis en premier, et comment (courriel piégé, page d’hameçonnage) ?
Depuis quelle adresse IP et quel pays l’attaquant s’est-il connecté ? À quelle heure ?

In [ ]:
# definition de compromissions
# repérer l'anomalie
# cadrer le périmétre de l'anomalie

 ### Niveau 3 — Suivre la progression

Repérez le balayage interne : quelle adresse, quels ports, quand ? Pourquoi cette adresse est-elle suspecte ?
Quel compte à privilèges a été compromis, et par quelle méthode ?
Quel compte l’attaquant a-t-il créé pour se maintenir, et à quel groupe l’a-t-il ajouté ?


### Niveau 4 — Mesurer l’impact

Y a-t-il eu exfiltration ? Vers quel hôte, quel jour, et combien de données (en Go) sont sorties ?
Quelles alertes de l’EDR auraient dû déclencher une réaction, et pourquoi sont-elles passées inaperçues ?


### Niveau 5 — Conclure


Établissez la chronologie complète de l’attaque et la liste des indicateurs de compromission.
Proposez des règles de détection qui auraient permis de repérer l’attaque plus tôt, et des mesures correctives.

### **Outils**

In [ ]:
# Exporter les dépendances
%pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
